In [ ]:
import pandas as pd
df = pd.read_csv('/content/netflix_titles.csv')
print(df.shape)
df.head()

(8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


### Uploading files to Colab
You can use `google.colab.files` to upload files from your local file system to the Colab environment. Once uploaded, these files are temporarily available in the Colab instance.

Here's an example:

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')


Saving netflix_titles.csv to netflix_titles.csv
User uploaded file "netflix_titles.csv" with length 3399671 bytes


After running the cell, a 'Choose Files' button will appear. Click on it to select the file(s) from your local computer that you want to upload. The files will be uploaded to the current Colab runtime's `/content/` directory by default.

In [ ]:
import pandas as pd
df = pd.read_csv('/content/netflix_titles.csv')
print(df.shape)
df.head()

(8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [ ]:
df.isnull().sum()

,0
show_id,0
type,0
title,0
director,2634
cast,825
country,831
date_added,10
release_year,0
rating,4
duration,3


In [ ]:
# 1. Duplicate rows remove pannunga
df.drop_duplicates(inplace=True)

# 2. Missing values handle pannunga
df['director'].fillna('Not Specified', inplace=True)
df['cast'].fillna('Not Specified', inplace=True)
df['country'].fillna('Not Specified', inplace=True)
df['rating'].fillna('Not Rated', inplace=True)
df['date_added'].fillna('Not Available', inplace=True)

# 3. Duration column split pannunga (movies ku minutes, TV shows ku seasons)
df['duration'] = df['duration'].fillna('Not Specified')

# 4. date_added column ah proper date format ku maathunga
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

# 5. release_year column check pannunga (already numeric ah irukkum)
df['release_year'] = pd.to_numeric(df['release_year'], errors='coerce')

# Verify cleaning
print(df.shape)
df.isnull().sum()

(8807, 12)


/tmp/ipykernel_478/3496726947.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['director'].fillna('Not Specified', inplace=True)
/tmp/ipykernel_478/3496726947.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', tr

,0
show_id,0
type,0
title,0
director,0
cast,0
country,0
date_added,98
release_year,0
rating,0
duration,0


In [ ]:
import sqlite3

# Create in-memory SQL database
conn = sqlite3.connect(':memory:')
df.to_sql('netflix', conn, index=False, if_exists='replace')

# Query 1: Movie vs TV Show count
query1 = """
SELECT type, COUNT(*) as total_count
FROM netflix
GROUP BY type
"""
result1 = pd.read_sql(query1, conn)
print(result1)

# Query 2: Top 10 countries by content count
query2 = """
SELECT country, COUNT(*) as total_shows
FROM netflix
WHERE country != 'Not Specified'
GROUP BY country
ORDER BY total_shows DESC
LIMIT 10
"""
result2 = pd.read_sql(query2, conn)
print(result2)

# Query 3: Content count by release year
query3 = """
SELECT release_year, COUNT(*) as total_shows
FROM netflix
GROUP BY release_year
ORDER BY release_year DESC
LIMIT 10
"""
result3 = pd.read_sql(query3, conn)
print(result3)

# Query 4: Rating distribution
query4 = """
SELECT rating, COUNT(*) as total_shows
FROM netflix
GROUP BY rating
ORDER BY total_shows DESC
"""
result4 = pd.read_sql(query4, conn)
print(result4)

      type  total_count
0    Movie         6131
1  TV Show         2676
          country  total_shows
0   United States         2818
1           India          972
2  United Kingdom          419
3           Japan          245
4     South Korea          199
5          Canada          181
6           Spain          145
7          France          124
8          Mexico          110
9           Egypt          106
   release_year  total_shows
0          2021          592
1          2020          953
2          2019         1030
3          2018         1147
4          2017         1032
5          2016          902
6          2015          560
7          2014          352
8          2013          288
9          2012          237
       rating  total_shows
0       TV-MA         3207
1       TV-14         2160
2       TV-PG          863
3           R          799
4       PG-13          490
5       TV-Y7          334
6        TV-Y          307
7          PG          287
8        TV-G          22

In [ ]:
df.to_csv('netflix_cleaned.csv', index=False)
